# UDFs and Pandas UDFs: When Built-in Functions Aren't Enough

Spark's `pyspark.sql.functions` library covers ~90% of real-world transformations. But occasionally
you need custom logic — a proprietary scoring algorithm, a regex that evolves weekly, a call to an
external library. That's when **UDFs** (User-Defined Functions) enter the picture.

There are three UDF variants in PySpark, with very different performance profiles:

---

## UDF Type Comparison

| Type | API | Execution | Performance | Use when |
|------|-----|-----------|-------------|----------|
| **Python UDF** | `@udf` / `spark.udf.register` | Row-by-row, Python process per executor | Slowest (~3-10x slower than built-ins) | Quick prototyping, rare transformations |
| **Pandas UDF (Scalar)** | `@pandas_udf` + `PandasUDFType.SCALAR` | Arrow batch per partition | 3-5x faster than Python UDF | Vectorisable math, ML scoring |
| **Pandas UDF (Grouped Map)** | `applyInPandas` | Whole group as a DataFrame | Best for group-level custom logic | Custom aggregation, per-group ML fitting |

---

## Notebook Roadmap

| Section | Topic |
|---------|-------|
| 1 | SparkSession setup |
| 2 | Python UDF with `@udf` decorator |
| 3 | Why Python UDFs are slow (theory) |
| 4 | Timing Python UDF vs built-in function |
| 5 | Pandas UDF concepts (theory) |
| 6 | Pandas UDF — Scalar type |
| 7 | Pandas UDF — Grouped Map (`applyInPandas`) |


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, DoubleType, IntegerType,
    StructType, StructField
)
import pandas as pd
import time

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("UDFs and Pandas UDFs")
    .config("spark.sql.shuffle.partitions", "8")
    # Arrow is required for Pandas UDFs — it's on by default in Spark 3.x,
    # but being explicit avoids confusion.
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("SparkSession ready.")

In [ ]:
# ============================================================
# SECTION 2: Python UDF with @udf decorator
# ============================================================
# The @udf decorator registers a Python function as a Spark column expression.
# You MUST specify the return type — Spark needs this to plan the schema
# before executing any data.

# Sample dataset: product names with messy formatting.
products_data = [
    (1, "  LAPTOP PRO 15  ",  1299.99),
    (2, "coffee_maker_deluxe",   89.50),
    (3, "Headphones-PREMIUM",   199.00),
    (4, "DESK CHAIR - ergonomic", 450.00),
    (5, "blender_ultra",         59.99),
]

products_df = spark.createDataFrame(
    products_data,
    schema=["id", "raw_name", "price"]
)

# --- Define a Python UDF ---
# This function normalises a product name:
# 1. Strip whitespace
# 2. Replace underscores and hyphens with spaces
# 3. Title-case the result
@F.udf(returnType=StringType())  # return type is mandatory
def normalise_product_name(name: str) -> str:
    """Normalise messy product names to Title Case."""
    if name is None:
        return None  # always handle nulls — Spark passes Python None for SQL null
    cleaned = name.strip()              # remove leading/trailing whitespace
    cleaned = cleaned.replace("_", " ") # underscores → spaces
    cleaned = cleaned.replace("-", " ") # hyphens → spaces
    return cleaned.title()              # Title Case

# Apply the UDF just like any built-in column expression.
udf_result_df = products_df.withColumn(
    "clean_name",
    normalise_product_name(F.col("raw_name"))  # UDF called with a Column argument
)
print("=== Python UDF result ===")
udf_result_df.show(truncate=False)

## Why Python UDFs Are Slow

Every time Spark calls a Python UDF, it has to cross the **JVM ↔ Python boundary** for each row:

```
JVM Executor                             Python Worker Process
────────────────────────                 ────────────────────────
Row 1 bytes → serialize to pickle   ──►  Python function(row_1)  → result_1
Row 2 bytes → serialize to pickle   ──►  Python function(row_2)  → result_2
...                                      ...
Row N bytes → serialize to pickle   ──►  Python function(row_N)  → result_N
                                    ◄──  deserialize from pickle
```

### The Four Cost Sources

1. **Serialization / Deserialization (serde)**: Every row is pickled from the JVM's binary format
   to Python's object model and back. Pickle is fast for small objects but adds up over millions
   of rows.

2. **Process startup**: Spark spawns a separate Python process per executor (actually per task),
   waiting for it to be ready.

3. **No Catalyst optimisation**: Catalyst can see *inside* built-in functions and reorder, fuse,
   or eliminate them. A Python UDF is a black box — Catalyst can only call it, never optimise it.

4. **No code generation (Tungsten)**: Built-in functions participate in Tungsten's whole-stage code
   generation, producing tight JVM bytecode loops. Python UDFs break the code-gen boundary.

### The Fix: Use Built-ins or Pandas UDFs

- If a `pyspark.sql.functions` equivalent exists → **always use it**.
- If you need custom Python logic → **Pandas UDF** (next section) avoids per-row overhead
  by batching rows into Arrow-backed pandas Series.


In [ ]:
# ============================================================
# SECTION 4: Demonstrating UDF Overhead — Timing Comparison
# ============================================================
# We'll run the same logic two ways:
#   A) Python UDF (slow path)
#   B) Equivalent pyspark.sql.functions chain (fast path)
# and time each with time.time().

# Generate a larger dataset so the difference is visible.
# spark.range(N) efficiently creates a DataFrame with a single 'id' column.
BIG_N = 500_000
big_df = (
    spark.range(BIG_N)
    .withColumn("raw_text", F.concat(F.lit("product_"), F.col("id").cast("string")))
)
big_df.cache()  # materialise once so both timings measure only the transformation
big_df.count()  # trigger the cache

# --- Path A: Python UDF ---
@F.udf(returnType=StringType())
def udf_title(text: str) -> str:
    """Title-case after replacing underscores."""
    if text is None:
        return None
    return text.replace("_", " ").title()

start_udf = time.time()
count_udf = big_df.withColumn("title", udf_title(F.col("raw_text"))).count()
time_udf = time.time() - start_udf

# --- Path B: Built-in functions ---
start_builtin = time.time()
count_builtin = (
    big_df
    .withColumn(
        "title",
        F.initcap(F.regexp_replace(F.col("raw_text"), "_", " "))  # equivalent logic
    )
    .count()
)
time_builtin = time.time() - start_builtin

print(f"Rows processed: {count_udf:,} (both paths)")
print(f"Python UDF time:    {time_udf:.2f}s")
print(f"Built-in func time: {time_builtin:.2f}s")
print(f"Built-in speedup:   {time_udf / time_builtin:.1f}x faster")

big_df.unpersist()  # free the cached data

## Pandas UDF (Vectorized UDF) — Apache Arrow to the Rescue

Pandas UDFs (introduced in Spark 2.3, significantly improved in 3.x) solve the per-row overhead
by processing **entire column batches** in one Python call using **Apache Arrow**.

### How Arrow Eliminates the Serde Bottleneck

```
Without Arrow (Python UDF)          With Arrow (Pandas UDF)
───────────────────────────         ───────────────────────────────────
JVM: row1 → pickle()                JVM: entire partition column →
JVM: row2 → pickle()                     Arrow IPC format (zero-copy)
...                                 Python: receives pandas.Series
Python: unpickle() row1             Python: applies vectorized ops
Python: unpickle() row2             Python: returns pandas.Series
...                                 JVM: Arrow IPC → back to JVM column
JVM: pickle() result1
JVM: pickle() result2
...
```

Arrow uses a **columnar, zero-copy memory format** that both JVM and Python understand directly —
no serialization overhead. The whole column crosses the boundary as a pointer, not a copy.

### Pandas UDF Types

| Type | Decorator | Input → Output | Frame size |
|------|-----------|---------------|------------|
| Scalar | `@pandas_udf(returnType)` | `Series → Series` | One Arrow batch per partition chunk |
| Scalar Iterator | `@pandas_udf(returnType)` with iterator signature | `Iterator[Series] → Iterator[Series]` | One batch at a time — good for expensive setup (model loading) |
| Grouped Map | `applyInPandas(func, schema)` | `DataFrame → DataFrame` | One call per group (entire group in memory) |

The **batch size** is controlled by `spark.sql.execution.arrow.maxRecordsPerBatch` (default: 10,000).
Increase it for throughput; decrease it if individual batches OOM.


In [ ]:
# ============================================================
# SECTION 6: Pandas UDF — SCALAR type
# ============================================================
# @pandas_udf with a returnType string/type treats the function as:
#   pandas.Series → pandas.Series
# Spark splits each partition into Arrow batches and calls the function
# once per batch, not once per row.

from pyspark.sql.functions import pandas_udf

# Rebuild the products DataFrame for a clean demo.
products_df2 = spark.createDataFrame(
    [
        (1, "  LAPTOP PRO 15  ",  1299.99),
        (2, "coffee_maker_deluxe",   89.50),
        (3, "Headphones-PREMIUM",   199.00),
        (4, "DESK CHAIR - ergonomic", 450.00),
        (5, "blender_ultra",         59.99),
        (6, None,                    49.00),  # null input — must be handled
    ],
    schema=["id", "raw_name", "price"]
)

# Scalar Pandas UDF: same logic as the Python UDF, but operates on a
# pandas.Series — all rows in the Arrow batch at once.
@pandas_udf(StringType())  # return type as a Spark DataType
def pandas_normalise_name(s: pd.Series) -> pd.Series:
    """
    Normalise product name using vectorised pandas string methods.
    pandas str methods automatically propagate NaN (None) — no explicit null check needed.
    """
    return (
        s.str.strip()           # remove leading/trailing whitespace
         .str.replace("_", " ", regex=False)  # underscores → spaces
         .str.replace("-", " ", regex=False)  # hyphens → spaces
         .str.title()           # Title Case (None stays None via pandas NaN propagation)
    )

# Also define a Pandas UDF for a numeric transformation — price category.
@pandas_udf(StringType())
def price_tier(prices: pd.Series) -> pd.Series:
    """Classify prices into tiers using vectorised pandas cut()."""
    return pd.cut(
        prices,
        bins=[0, 100, 500, float("inf")],
        labels=["budget", "mid-range", "premium"],
        right=True
    ).astype(str)  # pandas Categorical → string for Spark StringType

pandas_result_df = products_df2.withColumn(
    "clean_name", pandas_normalise_name(F.col("raw_name"))
).withColumn(
    "tier", price_tier(F.col("price"))
)

print("=== Scalar Pandas UDF result ===")
pandas_result_df.show(truncate=False)

In [ ]:
# ============================================================
# SECTION 7: applyInPandas — Group-Level Custom Computation
# ============================================================
# applyInPandas(func, schema) is the replacement for the deprecated
# GROUPED_MAP pandas_udf. It:
#   1. Groups the DataFrame by groupBy() keys
#   2. Collects each group into a pandas DataFrame
#   3. Calls your Python function with that pandas DataFrame
#   4. Expects a pandas DataFrame back (with the declared schema)
#
# IMPORTANT: the ENTIRE GROUP must fit in memory on one executor.
# Use this only when the group sizes are manageable.

# Use case: for each department, normalise salaries to Z-scores
# (mean=0, std=1). This requires knowing the group's mean and std
# simultaneously — it's much easier in pandas than with window functions.

# Employee data reused from the window-functions notebook context:
employee_data = [
    ("Engineering", "Alice",   95000),
    ("Engineering", "Bob",     82000),
    ("Engineering", "Charlie", 95000),
    ("Engineering", "Diana",   110000),
    ("Engineering", "Eve",     76000),
    ("Marketing",   "Frank",   68000),
    ("Marketing",   "Grace",   72000),
    ("Marketing",   "Hank",    68000),
    ("Marketing",   "Irene",   85000),
    ("HR",          "Jake",    55000),
    ("HR",          "Kate",    62000),
    ("HR",          "Leo",     58000),
]

emp_df = spark.createDataFrame(
    employee_data,
    schema=["dept", "name", "salary"]
)

# Output schema MUST be declared upfront — applyInPandas is not schema-inferring.
zscore_schema = StructType([
    StructField("dept",         StringType(),  False),
    StructField("name",         StringType(),  False),
    StructField("salary",       IntegerType(), False),
    StructField("salary_zscore",DoubleType(),  True),  # z-score can be NaN if std=0
])

def compute_zscore(pdf: pd.DataFrame) -> pd.DataFrame:
    """
    Receives one department's data as a pandas DataFrame.
    Returns the same DataFrame with a 'salary_zscore' column added.

    z = (x - mean) / std
    If std == 0 (all salaries identical), z-score is undefined → NaN.
    """
    mean_sal = pdf["salary"].mean()
    std_sal  = pdf["salary"].std(ddof=1)  # sample std (ddof=1), matches SQL STDDEV()

    if std_sal == 0 or pd.isna(std_sal):
        pdf["salary_zscore"] = float("nan")  # can't normalise a constant series
    else:
        pdf["salary_zscore"] = (pdf["salary"] - mean_sal) / std_sal

    pdf["salary_zscore"] = pdf["salary_zscore"].round(3)  # round to 3 decimal places
    return pdf[["dept", "name", "salary", "salary_zscore"]]

# Apply the function per-department.
# groupBy().applyInPandas() triggers a shuffle to co-locate each group on one executor.
zscore_df = emp_df.groupBy("dept").applyInPandas(compute_zscore, schema=zscore_schema)

print("=== Z-scored salaries per department ===")
zscore_df.orderBy("dept", "salary").show()

# Verify: within each dept the mean z-score should be ~0 and std ~1.
print("=== Verification: mean and std of z-scores per department ===")
zscore_df.groupBy("dept").agg(
    F.round(F.avg("salary_zscore"), 6).alias("mean_z"),   # should be ~0.0
    F.round(F.stddev("salary_zscore"), 6).alias("std_z"), # should be ~1.0
).orderBy("dept").show()

In [ ]:
# Clean shutdown.
spark.stop()
print("SparkSession stopped.")